# Structure Formation and LSS

1. **Lagrangian Perturbation Theory (LPT)** — Gaussian Random Field, 1LPT vs 2LPT vs 3LPT comparison, redshift evolution, power spectrum validation
2. **GRF Simulations** — standard, fixed-amplitude, and paired

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import toolscosmo as tcm
import tools21cm as t2c

plt.rcParams.update({'font.size': 12, 'lines.linewidth': 2})

---
## 1. Lagrangian Perturbation Theory (LPT)

### 1a. Setup and Gaussian Random Field

In [ ]:
grid_size = 128 #512 #
box_size  = 500 #Mpc/h 
param = tcm.par()
param.file.ps = 'CAMB' # 'CLASS' 'CLASSemu'
param.file.ps = tcm.get_Plin(param)
iSeed = 314159265

z1 = 149.
z2 = 9.

delta_lin = tcm.generate_gaussian_random_field(grid_size, box_size, param=param, random_seed=iSeed)['delta_lin']
particle_pos_z1 = tcm.generate_initial_condition_positions(grid_size, box_size, z1, param, LPT=2, delta_lin=delta_lin)['positions']
particle_pos_z2 = tcm.generate_initial_condition_positions(grid_size, box_size, z2, param, LPT=2, delta_lin=delta_lin)['positions']
delta_z1 = tcm.particles_on_grid(particle_pos_z1, grid_size, box_size, deconvolve=True, anti_aliasing=True)
delta_z2 = tcm.particles_on_grid(particle_pos_z2, grid_size, box_size, deconvolve=True, anti_aliasing=True)

fig, axs = plt.subplots(1,3,figsize=(12,4))
axs[0].set_title('Gaussian Random Field')
xx = np.linspace(0,box_size,grid_size)
axs[0].pcolor(xx, xx, delta_lin[:,:,grid_size//2])
xx = np.linspace(0,box_size,grid_size)
axs[1].set_title(f'$z={z1}$')
axs[1].pcolor(xx, xx, delta_z1[:,:,grid_size//2])
axs[2].set_title(f'$z={z2}$')
axs[2].pcolor(xx, xx, delta_z2[:,:,grid_size//2])
for ax in axs.flatten():
    ax.set_xlabel('[Mpc/$h$]')
    ax.set_ylabel('[Mpc/$h$]')
plt.tight_layout()
plt.show()

ps0 = t2c.power_spectrum_1d(delta_lin, kbins=20, box_dims=box_size)
ps1 = t2c.power_spectrum_1d(delta_z1, kbins=20, box_dims=box_size)
ps2 = t2c.power_spectrum_1d(delta_z2, kbins=20, box_dims=box_size)
p0, k0 = ps0
p1, k1 = ps1[0]/tcm.growth_factor(z1, param)**2, ps1[1]
p2, k2 = ps2[0]/tcm.growth_factor(z2, param)**2, ps2[1]

fig, ax = plt.subplots(1,1,figsize=(5,4))
kk, pp = param.file.ps['k'], param.file.ps['P']
ax.loglog(kk, pp, c='k', ls='-', label='$P_\mathrm{{lin}}$')
ax.loglog(k0, p0, c='r', ls='--', label='$P^{{grid}}_\mathrm{{lin}}$')
ax.loglog(k1, p1*(grid_size/box_size)**-1, c='C0', ls='-.', label=f'$z={z1}$')
ax.loglog(k2, p2*(grid_size/box_size)**-1, c='C1', ls=':', label=f'$z={z2}$')
ax.set_xlabel('[$h$/Mpc]')
ax.set_ylabel('$P(k)$')
ax.axis([8e-3,3,3,4e4])
ax.legend()
plt.tight_layout()
plt.show()

### 1b. 1LPT vs 2LPT vs 3LPT comparison at fixed redshift

Each order adds a correction to the displacement field:
- **1LPT** (Zel'dovich): Ψ ~ D₁ ∇φ⁽¹⁾
- **2LPT**: adds D₂ ∇φ⁽²⁾, sourced by the tidal quadratic invariant G₂(φ⁽¹⁾)
- **3LPT**: adds D₃ₐ ∇φ⁽³ᵃ⁾ + D₃ᵦ ∇φ⁽³ᵇ⁾, where 3a is sourced by det(∂ᵢ∂ⱼφ⁽¹⁾) and 3b by the mixed invariant G₂(φ⁽¹⁾, φ⁽²⁾)

Growth factors (EdS approximations):
- D₂ ≈ −(3/7) D₁²
- D₃ₐ ≈ +(1/3) D₁³
- D₃ᵦ ≈ −(10/21) D₁ D₂ ≈ +(10/49) D₁³

MAS deconvolution (`deconvolve=True`) divides by the CIC window W(k) = sinc(k Δx/2)² to undo
kernel smoothing, while anti-aliasing (`anti_aliasing=True`) zeros modes above k_grid = N/3
(Orszag 2/3 rule) before deconvolution. Together they recover P(k) ≈ D²(z) P_lin up to k ~ N/(3L)
without the high-k amplification that deconvolution alone produces from shot noise and aliased modes.

In [ ]:
z_comp = 7.0   # comparison redshift

pos_1lpt = tcm.generate_initial_condition_positions(grid_size, box_size, z_comp, param, LPT=1, delta_lin=delta_lin)['positions']
pos_2lpt = tcm.generate_initial_condition_positions(grid_size, box_size, z_comp, param, LPT=2, delta_lin=delta_lin)['positions']
pos_3lpt = tcm.generate_initial_condition_positions(grid_size, box_size, z_comp, param, LPT=3, delta_lin=delta_lin)['positions']

delta_1lpt = tcm.particles_on_grid(pos_1lpt, grid_size, box_size, deconvolve=True, anti_aliasing=True)
delta_2lpt = tcm.particles_on_grid(pos_2lpt, grid_size, box_size, deconvolve=True, anti_aliasing=True)
delta_3lpt = tcm.particles_on_grid(pos_3lpt, grid_size, box_size, deconvolve=True, anti_aliasing=True)

k_nyq = np.pi * grid_size / box_size

fig, axs = plt.subplots(2, 2, figsize=(11, 10))
xx = np.linspace(0, box_size, grid_size)
sl = grid_size // 2

# Density field slices: 1LPT, 2LPT, 3LPT
for ax, field, title in zip([axs[0,0], axs[0,1], axs[1,0]],
                             [delta_1lpt, delta_2lpt, delta_3lpt],
                             [f'1LPT  $z={z_comp}$', f'2LPT  $z={z_comp}$', f'3LPT  $z={z_comp}$']):
    ax.pcolor(xx, xx, field[:, :, sl], cmap='magma')
    ax.set_title(title)
    ax.set_xlabel('[Mpc/$h$]')
    ax.set_ylabel('[Mpc/$h$]')

# Power spectrum comparison (bottom-right)
ps_1lpt = t2c.power_spectrum_1d(delta_1lpt, kbins=20, box_dims=box_size)
ps_2lpt = t2c.power_spectrum_1d(delta_2lpt, kbins=20, box_dims=box_size)
ps_3lpt = t2c.power_spectrum_1d(delta_3lpt, kbins=20, box_dims=box_size)

ax = axs[1, 1]
kk, pp = param.file.ps['k'], param.file.ps['P']
D_z = tcm.growth_factor(z_comp, param)
ax.loglog(kk, pp * D_z**2, c='k', ls='-',  lw=2, label='$D^2(z) P_\\mathrm{lin}$')
ax.loglog(ps_1lpt[1], ps_1lpt[0] * (grid_size / box_size)**-1, c='C0', ls='--', label='1LPT')
ax.loglog(ps_2lpt[1], ps_2lpt[0] * (grid_size / box_size)**-1, c='C1', ls='-.',  label='2LPT')
ax.loglog(ps_3lpt[1], ps_3lpt[0] * (grid_size / box_size)**-1, c='C3', ls=':',   label='3LPT')
ax.axvline(k_nyq,     color='gray', ls=':',  lw=1.2,      label='$k_\\mathrm{Nyq}$')
ax.axvline(k_nyq / 2, color='gray', ls='--', lw=0.8, alpha=0.6, label='$k_\\mathrm{Nyq}/2$')
ax.set_xlabel('[$h$/Mpc]')
ax.set_ylabel('$P(k)$')
ax.set_title(f'Power spectrum at $z={z_comp}$')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## 2. GRF Simulations: standard, fixed-amplitude, and paired

**Standard GRF** — amplitude and phase of each Fourier mode are independently random;
measured P(k) fluctuates around the input P(k) due to sample variance.

**Fixed-amplitude GRF** (Angulo & Pontzen 2016) — amplitude of each mode is set to
exactly sqrt(P(k)); only the phase is random. The measured P(k) of a single realisation
then matches the input exactly, dramatically reducing sample-variance noise.

**Paired GRF** — the partner simulation of a fixed-amplitude run, obtained by passing a
negative seed (`random_seed = -N` is the pair of `random_seed = N`). The density field is
simply sign-flipped: δ_paired = −δ_fixed. Averaging P(k) over a fixed+paired pair further
cancels odd-order mode-coupling terms.

In [ ]:
def simulate_delta_GRF(Om=0.315, Ob=0.049, Or=5.4e-05, As=2.089e-09, h0=0.673, ns=0.963,
                       iSeed=42, grid_size=128, box_size=500, fixed_amplitude=False):
    p = tcm.par()
    p.cosmo.Om = Om; p.cosmo.Ob = Ob; p.cosmo.Or = Or; p.cosmo.Ok = 0.
    p.cosmo.As = As; p.cosmo.h0 = h0; p.cosmo.ns = ns
    p.file.ps = tcm.get_Plin(p)
    delta = tcm.generate_gaussian_random_field(grid_size, box_size, param=p,
                random_seed=iSeed, fixed_amplitude=fixed_amplitude)['delta_lin']
    return {'delta_lin': delta, 'PS': p.file.ps}

# --- 3a. Fixed and paired GRF demo ---
grid_size = 128
box_size  = 500   # Mpc/h
iSeed     = 42

grf_std   = simulate_delta_GRF(iSeed= iSeed, grid_size=grid_size, box_size=box_size, fixed_amplitude=False)
grf_fixed = simulate_delta_GRF(iSeed= iSeed, grid_size=grid_size, box_size=box_size, fixed_amplitude=True)
grf_pair  = simulate_delta_GRF(iSeed=-iSeed, grid_size=grid_size, box_size=box_size, fixed_amplitude=True)

ps_std,   ks = t2c.power_spectrum_1d(grf_std['delta_lin'],   kbins=20, box_dims=box_size)
ps_fixed, _  = t2c.power_spectrum_1d(grf_fixed['delta_lin'], kbins=20, box_dims=box_size)
ps_pair,  _  = t2c.power_spectrum_1d(grf_pair['delta_lin'],  kbins=20, box_dims=box_size)
ps_avg        = (ps_fixed + ps_pair) / 2

kk, pp = grf_std['PS']['k'], grf_std['PS']['P']

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

sl = grid_size // 2
xx = np.linspace(0, box_size, grid_size)
axes[0].pcolor(xx, xx, grf_fixed['delta_lin'][:, :, sl], cmap='RdBu_r')
axes[0].set_title('Fixed-amplitude GRF')
axes[0].set_xlabel('[Mpc/$h$]'); axes[0].set_ylabel('[Mpc/$h$]')

ax = axes[1]
ax.loglog(kk, pp,       'k-',  lw=2.5, label='Input $P_\\mathrm{lin}$')
ax.loglog(ks, ps_std,   c='C2', ls=':',  label='Standard (random amp)')
ax.loglog(ks, ps_fixed, c='C0', ls='--', label='Fixed amplitude')
ax.loglog(ks, ps_pair,  c='C1', ls='-.', label='Paired (−seed)')
ax.loglog(ks, ps_avg,   c='C3', ls='-',  lw=2, label='(Fixed + Paired) / 2')
ax.set_xlabel('$k$ [$h$/Mpc]'); ax.set_ylabel('$P(k)$')
ax.set_title('Power spectrum: fixed vs standard')
ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# --- 3b. 21cm GRF with cosmological parameter variations ---
def simulate_21cm_GRF(z=9., Om=0.315, Ob=0.049, Or=5.4e-05, As=2.089e-09, h0=0.673, ns=0.963,
                      iSeed=42, grid_size=128, box_size=500):
    grf = simulate_delta_GRF(Om=Om, Ob=Ob, Or=Or, As=As, h0=h0, ns=ns,
                             iSeed=iSeed, grid_size=grid_size, box_size=box_size)
    grf['dt21'] = t2c.mean_dt(z) * (1 + grf['delta_lin'])
    return grf

Om_list    = [0.27, 0.30]
iSeed_list = [42, 64]

fig, axs = plt.subplots(2, 3, figsize=(13, 8))
xx = np.linspace(0, box_size, grid_size)
for i, Om in enumerate(Om_list):
    for j, iSeed in enumerate(iSeed_list):
        grf = simulate_21cm_GRF(Om=Om, iSeed=iSeed, grid_size=grid_size, box_size=box_size)
        ps, ks = t2c.power_spectrum_1d(grf['delta_lin'], kbins=15, box_dims=box_size)
        axs[i, j].set_title(r'$\Omega_m=%.2f$, seed=%d' % (Om, iSeed))
        im = axs[i, j].pcolor(xx, xx, grf['delta_lin'][10])
        axs[i, j].set_xlabel('X [Mpc/h]'); axs[i, j].set_ylabel('Y [Mpc/h]')
        fig.colorbar(im, ax=axs[i, j])
        axs[i, -1].loglog(ks, ps * ks**3 / 2 / np.pi**2, c=f'C{j}', label='seed=%d' % iSeed, zorder=3)
    ps, ks = grf['PS']['P'], grf['PS']['k']
    axs[i, -1].loglog(ks, ps * ks**3 / 2 / np.pi**2, c='k', label='CAMB', zorder=1)
    axs[i, -1].set_title(r'$\Omega_m=%.2f$' % Om)
    axs[i, -1].axis([8e-3, 3, 3e-3, 4])
    axs[i, -1].set_xlabel('k [h/Mpc]'); axs[i, -1].set_ylabel(r'$\Delta^2$ [h/Mpc]')
    axs[i, -1].legend()
plt.tight_layout(); plt.show()